# 第１弾 STEP 2: Iceberg Table の読み取り・スナップショット・タイムトラベル

## 目的
- Iceberg Table からデータを読み取る
- スナップショットを使ったタイムトラベルを確認する
- 特定スナップショット時点のデータを参照する
- Iceberg の変更履歴（change tracking）の仕組みを理解する

## 前提
`01_write_and_structure.ipynb` を先に実行して `ORDERS_ICEBERG` テーブルにデータが入っていること

## STEP 0: 接続確認

In [ ]:
USE DATABASE ICEBERG_DB;
USE SCHEMA WORK;
USE WAREHOUSE SANDBOX_WH;

## STEP 1: 通常の読み取り

In [ ]:
-- 通常クエリ（最新スナップショットを参照）
SELECT
    YEAR(O_ORDERDATE) AS ORDER_YEAR,
    COUNT(*) AS CNT,
    SUM(O_TOTALPRICE) AS TOTAL_PRICE
FROM ICEBERG_DB.WORK.ORDERS_ICEBERG
GROUP BY 1
ORDER BY 1;

## STEP 2: スナップショット一覧の確認

In [ ]:
-- スナップショット一覧（SNAPSHOT_ID と COMMITTED_AT を控えておく）
SELECT
    SNAPSHOT_ID,
    COMMITTED_AT,
    OPERATION,
    SUMMARY
FROM TABLE(
    INFORMATION_SCHEMA.ICEBERG_TABLE_SNAPSHOTS(
        TABLE_NAME => 'ORDERS_ICEBERG',
        TABLE_SCHEMA => 'WORK',
        TABLE_DATABASE => 'ICEBERG_DB'
    )
)
ORDER BY COMMITTED_AT;

## STEP 3: タイムトラベル（スナップショット指定）

Iceberg Table では `AT (SNAPSHOT => '<snapshot_id>')` または  
`AT (TIMESTAMP => '<timestamp>')` で過去時点のデータを参照できる。

In [ ]:
-- 最初のスナップショット（1回目INSERTの直後）を参照
-- ★ 上のセルで取得した最初の SNAPSHOT_ID に書き換えること
SELECT COUNT(*)
FROM ICEBERG_DB.WORK.ORDERS_ICEBERG
AT (SNAPSHOT => '<最初のSNAPSHOT_ID>');

In [ ]:
-- タイムスタンプ指定でタイムトラベル
-- ★ 1回目INSERT完了後のタイムスタンプに書き換えること
SELECT COUNT(*)
FROM ICEBERG_DB.WORK.ORDERS_ICEBERG
AT (TIMESTAMP => '<1回目INSERT完了直後のタイムスタンプ>'::TIMESTAMP_LTZ);

## STEP 4: 変更履歴の確認（CHANGES 句）

2つのスナップショット間の変更行を取得できる。

In [ ]:
-- Iceberg Table に CHANGE TRACKING を有効化
ALTER TABLE ICEBERG_DB.WORK.ORDERS_ICEBERG SET CHANGE_TRACKING = TRUE;

In [ ]:
-- 2つのスナップショット間の変更を確認
-- ★ SNAPSHOT_ID は上のセルで取得した値に書き換えること
SELECT
    METADATA$ACTION,      -- INSERT / DELETE
    METADATA$ISUPDATE,    -- UPDATE の場合は true
    O_ORDERKEY,
    O_ORDERSTATUS
FROM ICEBERG_DB.WORK.ORDERS_ICEBERG
CHANGES(INFORMATION => APPEND_ONLY)
AT (SNAPSHOT => '<1つ目のSNAPSHOT_ID>')
END (SNAPSHOT => '<最新のSNAPSHOT_ID>')
LIMIT 20;

## STEP 5: Iceberg Table のメタデータ詳細確認

In [ ]:
-- テーブル情報の詳細（S3 上のメタデータパスが確認できる）
SELECT PARSE_JSON(
    SYSTEM$ICEBERG_TABLE_INFO('ICEBERG_DB.WORK.ORDERS_ICEBERG')
);

In [ ]:
-- メタデータの各フィールドを展開して確認
WITH info AS (
    SELECT PARSE_JSON(
        SYSTEM$ICEBERG_TABLE_INFO('ICEBERG_DB.WORK.ORDERS_ICEBERG')
    ) AS j
)
SELECT
    j:status::STRING            AS status,
    j:metadataLocation::STRING  AS metadata_location,
    j:catalogType::STRING       AS catalog_type
FROM info;

## STEP 6: テーブル統計の確認

In [ ]:
-- テーブルのストレージ使用量
SELECT
    TABLE_NAME,
    ROW_COUNT,
    BYTES,
    CLUSTERING_KEY,
    CREATED
FROM ICEBERG_DB.INFORMATION_SCHEMA.TABLES
WHERE TABLE_NAME = 'ORDERS_ICEBERG';

## まとめ・気づき

- `AT (SNAPSHOT => ...)` でスナップショット単位のタイムトラベルが可能
- `AT (TIMESTAMP => ...)` でタイムスタンプ指定のタイムトラベルも可能
- `CHANGES` 句で2時点間の差分（INSERT/DELETE/UPDATE）を取得できる
- `SYSTEM$ICEBERG_TABLE_INFO()` で S3 上の metadata ファイルの場所が確認できる
- Snowflake Managed Iceberg は Snowflake が catalog（スナップショット管理）を担当する